# Lab 2 — Multi-seed evaluation of the three winning models

<a href="https://colab.research.google.com/github/lgonperez3-glitch/MLSM-labs/blob/main/Lab2/lab2_multiseed_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook checks how stable the comparison between the three selected models is when the train/validation/test split changes.

It **re-trains** the winning ANN, CNN and VGG16 configurations from scratch for each seed. The three models use the same stratified split for a given seed. The hyperparameter configurations remain fixed: this notebook does not repeat the 16-experiment search.

**Models and fixed configurations**
- Lab 1 ANN: `MLPClassifier(hidden_layer_sizes=(128, 64))`.
- Lab 2 Task 1 CNN: `size_192x288`, with the winning CNN configuration.
- Lab 2 Task 2 VGG16: `size_192x288`, with the winning VGG16 configuration (ImageNet weights, first four blocks frozen).

**Seeds:** 42, 7, 21, 100 and 2026.

The test set is used only for final evaluation in each run. Validation is used for early stopping in the neural networks. Five seeds provide a better view of variability than one split, but they are still a limited experiment and do not by themselves establish statistical significance.

Before running: mount the Google Drive containing `MLSM/Data/features_30_sec.csv` and `MLSM/Data/images_original`.

## 1. Mount Drive, paths and imports

The results are saved in a new folder, so the original Task 1, Task 2 and comparison results are not overwritten.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import re
import gc
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore", category=FutureWarning)

CSV_PATH = Path("/content/drive/MyDrive/MLSM/Data/features_30_sec.csv")
DATA_DIR = Path("/content/drive/MyDrive/MLSM/Data/images_original")
OUTPUT_DIR = Path("/content/drive/MyDrive/MLSM/Results/Lab 2/multiseed_evaluation_v2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 7, 21, 100, 2026]
BATCH_SIZE = 32
MAX_EPOCHS = 50
PATIENCE = 10
MIN_DELTA = 0.001

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV not found: {CSV_PATH}")
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Image directory not found: {DATA_DIR}")

print("TensorFlow:", tf.__version__)
print("Seeds:", SEEDS)
print("Output folder:", OUTPUT_DIR)

## 2. Load and match the image and CSV data

The image manifest follows the same ordering rule as the Task 1 and Task 2 notebooks: alphabetically sorted genre folders and filenames. Each image is matched to its CSV row through a normalized song ID. This is important because all three models must use exactly the same songs in each split.

In [ ]:
def canonical_song_id(filename):
    stem = Path(str(filename).replace("\\", "/")).name
    stem = Path(stem).stem
    return re.sub(r"[^a-zA-Z0-9]", "", stem).lower()

df = pd.read_csv(CSV_PATH)
required_columns = {"label", "filename"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Missing required CSV columns: {sorted(missing)}")

df = df.dropna(subset=["label", "filename"]).copy()
df["label"] = df["label"].astype(str).str.strip()
df["song_id"] = df["filename"].map(canonical_song_id)

if df["song_id"].duplicated().any():
    raise ValueError("Duplicate normalized song IDs found in the CSV.")

class_names = sorted(folder.name for folder in DATA_DIR.iterdir() if folder.is_dir())
valid_extensions = (".png", ".jpg", ".jpeg", ".bmp")

image_rows = []
for label_id, genre in enumerate(class_names):
    genre_folder = DATA_DIR / genre
    for image_path in sorted(genre_folder.iterdir(), key=lambda p: p.name):
        if image_path.is_file() and image_path.suffix.lower() in valid_extensions:
            image_rows.append({
                "image_path": str(image_path),
                "song_id": canonical_song_id(image_path.name),
                "genre": genre,
                "image_label": label_id
            })

image_manifest = pd.DataFrame(image_rows)
if image_manifest.empty:
    raise ValueError("No images found.")
if image_manifest["song_id"].duplicated().any():
    raise ValueError("Duplicate normalized song IDs found in the image folders.")

csv_ids_before_matching = set(df["song_id"])
image_ids_before_matching = set(image_manifest["song_id"])
common_ids = csv_ids_before_matching & image_ids_before_matching
csv_only_count = len(csv_ids_before_matching - image_ids_before_matching)
image_only_count = len(image_ids_before_matching - csv_ids_before_matching)
df = df[df["song_id"].isin(common_ids)].copy()
image_manifest = image_manifest[image_manifest["song_id"].isin(common_ids)].reset_index(drop=True)

if len(df) != len(image_manifest) or set(df["song_id"]) != set(image_manifest["song_id"]):
    raise ValueError("The CSV and image manifest do not contain the same songs.")

csv_genres = df.set_index("song_id")["label"]
image_genres = image_manifest.set_index("song_id")["genre"]
mismatches = [
    sid for sid in common_ids
    if csv_genres.loc[sid] != image_genres.loc[sid]
]
if mismatches:
    raise ValueError(f"Genre mismatch between CSV and image folders: {mismatches[:10]}")

feature_df = df.drop(columns=["label", "filename", "song_id"], errors="ignore")
feature_df = feature_df.select_dtypes(include=[np.number]).copy()
if feature_df.empty:
    raise ValueError("No numeric features found in the CSV.")

X_by_id = feature_df.copy()
X_by_id.index = df["song_id"].values

print("Genres:", class_names)
print("Common songs:", len(image_manifest))
print("Numeric features:", X_by_id.shape[1])
print("CSV rows without a matching image:", csv_only_count)
print("Images without a matching CSV row:", image_only_count)

## 3. Selección de las configuraciones ganadoras

CNN y VGG16 se seleccionan leyendo los mismos CSV de experimentos que utiliza `lab1vslab2_colab.ipynb`, con el mismo criterio: mayor `val_accuracy` y desempate por `name` ascendente. No se vuelven a ejecutar búsquedas de hiperparámetros.

La ANN conserva la configuración ganadora de Lab 1 definida en el notebook de comparación: capas ocultas (128, 64), Adam, `alpha=0.001`, `learning_rate_init=0.001`, `max_iter=10000`, sin early stopping, `n_iter_no_change=15` y `tol=0.001`.

Los CSV de Task 1 y Task 2 deben existir en Drive y contener los 16 experimentos esperados. La selección utiliza únicamente validación, nunca test.

In [ ]:
NUM_CLASSES = len(class_names)

# Las rutas y el criterio de selección replican lab1vslab2_colab.ipynb.
TASK1_RESULTS_DIR = Path("/content/drive/MyDrive/MLSM/Results/Lab 2/task1_results_v3")
TASK2_RESULTS_DIR = Path("/content/drive/MyDrive/MLSM/Results/Lab 2/task2_vgg16_results_v1")
TASK1_CSV = TASK1_RESULTS_DIR / "task1_two_stage_experiments.csv"
TASK2_CSV = TASK2_RESULTS_DIR / "task2_two_stage_experiments.csv"

for required_file in [TASK1_CSV, TASK2_CSV]:
    if not required_file.is_file():
        raise FileNotFoundError(
            f"No existe {required_file}. Ejecuta primero el notebook "
            "correspondiente de Lab 2 hasta completar sus 16 experimentos."
        )

task1_experiments = pd.read_csv(TASK1_CSV)
task2_experiments = pd.read_csv(TASK2_CSV)
if len(task1_experiments) != 16 or len(task2_experiments) != 16:
    raise ValueError(
        "Se esperaban 16 experimentos en cada tarea. "
        f"Task 1: {len(task1_experiments)}; Task 2: {len(task2_experiments)}."
    )

# Mismo criterio de lab1vslab2_colab.ipynb.
task1_winner = task1_experiments.sort_values(
    ["val_accuracy", "name"], ascending=[False, True]
).iloc[0]
task2_winner = task2_experiments.sort_values(
    ["val_accuracy", "name"], ascending=[False, True]
).iloc[0]

def task1_config_from_row(row):
    return {
        "filters": tuple(map(int, str(row["filters"]).split("-"))),
        "kernel": tuple(map(int, str(row["kernel"]).split("x"))),
        "dropout": float(row["dropout"]),
        "augment": str(row["augment"]).lower() == "true",
        "lr": float(row["lr"]),
        "size": (int(row["img_h"]), int(row["img_w"])),
    }

def task2_config_from_row(row):
    return {
        "filters": tuple(map(int, str(row["filters"]).split("-"))),
        "vgg_blocks": int(row["vgg_blocks"]),
        "kernel": tuple(map(int, str(row["kernel"]).split("x"))),
        "dropout": float(row["dropout"]),
        "augment": str(row["augment"]).lower() == "true",
        "lr": float(row["lr"]),
        "size": (int(row["img_h"]), int(row["img_w"])),
    }

CNN_CONFIG = task1_config_from_row(task1_winner)
VGG16_CONFIG = task2_config_from_row(task2_winner)

# Configuración ANN ganadora de Lab 1, igual que en el notebook comparativo.
ANN_CONFIG = {
    "hidden_layer_sizes": (128, 64),
    "solver": "adam",
    "learning_rate_init": 0.001,
    "alpha": 0.001,
    "max_iter": 10000,
    "early_stopping": False,
    "n_iter_no_change": 15,
    "tol": 1e-3,
}

winner_summary = pd.DataFrame([
    {"Model": "ANN", "Winning experiment": "Lab 1 fixed winning MLP",
     "Validation accuracy": np.nan, "Configuration": str(ANN_CONFIG)},
    {"Model": "CNN", "Winning experiment": str(task1_winner["name"]),
     "Validation accuracy": float(task1_winner["val_accuracy"]),
     "Configuration": str(CNN_CONFIG)},
    {"Model": "VGG16", "Winning experiment": str(task2_winner["name"]),
     "Validation accuracy": float(task2_winner["val_accuracy"]),
     "Configuration": str(VGG16_CONFIG)},
])
display(winner_summary)

# Registro para auditar qué configuración se utilizó en las 15 ejecuciones.
winning_configs = {
    "selection_rule": "Highest val_accuracy; ties resolved by ascending experiment name",
    "cnn_winner": str(task1_winner["name"]),
    "cnn_val_accuracy": float(task1_winner["val_accuracy"]),
    "cnn_config": CNN_CONFIG,
    "vgg16_winner": str(task2_winner["name"]),
    "vgg16_val_accuracy": float(task2_winner["val_accuracy"]),
    "vgg16_config": VGG16_CONFIG,
    "ann_source": "Winning Lab 1 ANN configuration in lab1vslab2_colab.ipynb",
    "ann_config": ANN_CONFIG,
}
with open(OUTPUT_DIR / "winning_configurations.json", "w") as f:
    json.dump(winning_configs, f, indent=2)

print("CNN winner:", task1_winner["name"])
print("CNN validation accuracy:", round(float(task1_winner["val_accuracy"]), 4))
print("VGG16 winner:", task2_winner["name"])
print("VGG16 validation accuracy:", round(float(task2_winner["val_accuracy"]), 4))
print("Configuration audit file:", OUTPUT_DIR / "winning_configurations.json")

## 4. Create one shared split for each seed

For every seed, we first reserve 20% for test. Then, 20% of the remaining 80% is used for validation. This gives approximately 64% train, 16% validation and 20% test.

The same split is passed to the ANN, CNN and VGG16. The split is stratified by genre, and the song IDs are retained so that tabular rows and images stay aligned.

In [ ]:
image_paths = image_manifest["image_path"].to_numpy()
image_labels = image_manifest["image_label"].to_numpy()
image_song_ids = image_manifest["song_id"].to_numpy()

def create_seed_split(seed):
    dev_paths, test_paths, dev_labels, test_labels, dev_ids, test_ids = train_test_split(
        image_paths,
        image_labels,
        image_song_ids,
        test_size=0.20,
        random_state=seed,
        stratify=image_labels
    )

    train_paths, val_paths, train_labels, val_labels, train_ids, val_ids = train_test_split(
        dev_paths,
        dev_labels,
        dev_ids,
        test_size=0.20,
        random_state=seed,
        stratify=dev_labels
    )

    # Check that no song appears in more than one partition.
    assert set(train_ids).isdisjoint(set(val_ids))
    assert set(train_ids).isdisjoint(set(test_ids))
    assert set(val_ids).isdisjoint(set(test_ids))
    assert len(train_ids) + len(val_ids) + len(test_ids) == len(image_manifest)

    # Check that every ID has a corresponding row of numerical features.
    assert set(np.concatenate([train_ids, val_ids, test_ids])) == set(X_by_id.index)

    return {
        "train_paths": train_paths,
        "val_paths": val_paths,
        "test_paths": test_paths,
        "train_labels": train_labels.astype(int),
        "val_labels": val_labels.astype(int),
        "test_labels": test_labels.astype(int),
        "train_ids": train_ids,
        "val_ids": val_ids,
        "test_ids": test_ids,
    }

split_sizes = []
for seed in SEEDS:
    split = create_seed_split(seed)
    split_sizes.append({
        "Seed": seed,
        "Train": len(split["train_ids"]),
        "Validation": len(split["val_ids"]),
        "Test": len(split["test_ids"]),
    })
display(pd.DataFrame(split_sizes))

## 5. ANN: train the fixed MLP configuration

The preprocessing steps are fitted only on the training partition:
- median imputation handles missing numerical values;
- variance filtering removes constant features;
- standardization puts features on comparable scales;
- `SelectKBest(k="all")` keeps all remaining features, matching the comparison notebook;
- the MLP learns the classification boundaries.

The ANN does not use validation-based early stopping in the original setup, so the same setting is retained here.

In [ ]:
def build_ann(seed):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("variance", VarianceThreshold(threshold=0.0)),
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k="all")),
        ("model", MLPClassifier(
            hidden_layer_sizes=ANN_CONFIG["hidden_layer_sizes"],
            solver=ANN_CONFIG["solver"],
            learning_rate_init=ANN_CONFIG["learning_rate_init"],
            alpha=ANN_CONFIG["alpha"],
            max_iter=ANN_CONFIG["max_iter"],
            random_state=seed,
            early_stopping=ANN_CONFIG["early_stopping"],
            n_iter_no_change=ANN_CONFIG["n_iter_no_change"],
            tol=ANN_CONFIG["tol"]
        ))
    ])

## 6. Image loading and training datasets

Images are resized to the winning resolution. Training images are shuffled; validation and test images are not. The data augmentation is part of each model architecture and is active only during training.

In [ ]:
def load_image(path, label, size):
    image_bytes = tf.io.read_file(path)
    image = tf.image.decode_image(
        image_bytes,
        channels=3,
        expand_animations=False
    )
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, size)
    return image, label

def make_dataset(file_paths, file_labels, size, seed, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((file_paths, file_labels))
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(file_paths),
            seed=seed,
            reshuffle_each_iteration=True
        )
    dataset = dataset.map(
        lambda path, label: load_image(path, label, size),
        num_parallel_calls=tf.data.AUTOTUNE
    )
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

class SaveBestWeights(tf.keras.callbacks.Callback):
    """Keep the weights from the epoch with the highest validation accuracy."""
    def on_train_begin(self, logs=None):
        self.best_val_accuracy = -1.0
        self.best_weights = None
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        val_accuracy = (logs or {}).get("val_accuracy")
        if val_accuracy is not None and val_accuracy > self.best_val_accuracy:
            self.best_val_accuracy = float(val_accuracy)
            self.best_weights = [weight.copy() for weight in self.model.get_weights()]
            self.best_epoch = epoch + 1

def compile_model(model, learning_rate, n_train):
    steps_per_epoch = int(np.ceil(n_train / BATCH_SIZE))
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=learning_rate,
        decay_steps=max(1, steps_per_epoch * MAX_EPOCHS),
        alpha=0.0
    )
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=lr_schedule,
            weight_decay=1e-4
        ),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def fit_with_validation(model, train_ds, val_ds):
    save_best = SaveBestWeights()
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=PATIENCE,
        min_delta=MIN_DELTA,
        restore_best_weights=True
    )
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=MAX_EPOCHS,
        shuffle=False,
        callbacks=[save_best, early_stopping],
        verbose=1
    )
    if save_best.best_weights is None:
        raise RuntimeError("Training did not produce a valid best checkpoint.")
    model.set_weights(save_best.best_weights)
    return history, save_best.best_epoch

## 7. Build the CNN and VGG16 models

The CNN uses the same convolutional blocks as Task 1. VGG16 uses ImageNet weights and keeps the first four pretrained blocks frozen, as in Task 2. Only the additional convolutional layer and classification head are trained.

The VGG16 ImageNet weights may need to be downloaded the first time this cell is run.

In [ ]:
def build_cnn(seed, n_train):
    tf.keras.utils.set_random_seed(seed)
    height, width = CNN_CONFIG["size"]
    inputs = tf.keras.Input(shape=(height, width, 3))
    x = inputs

    if CNN_CONFIG["augment"]:
        x = tf.keras.layers.RandomTranslation(
            height_factor=0.0, width_factor=0.05
        )(x)
        x = tf.keras.layers.RandomZoom(
            height_factor=0.0, width_factor=0.1
        )(x)
        x = tf.keras.layers.RandomContrast(0.1)(x)

    x = tf.keras.layers.Rescaling(1.0 / 255.0)(x)

    for n_filters in CNN_CONFIG["filters"]:
        x = tf.keras.layers.Conv2D(
            filters=n_filters,
            kernel_size=CNN_CONFIG["kernel"],
            padding="same",
            use_bias=False
        )(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.Activation("relu")(x)
        x = tf.keras.layers.MaxPooling2D(pool_size=2)(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    if CNN_CONFIG["dropout"] > 0:
        x = tf.keras.layers.Dropout(CNN_CONFIG["dropout"])(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    return compile_model(model, CNN_CONFIG["lr"], n_train)

def build_vgg16(seed, n_train):
    tf.keras.utils.set_random_seed(seed)
    height, width = VGG16_CONFIG["size"]
    inputs = tf.keras.Input(shape=(height, width, 3))
    x = inputs

    if VGG16_CONFIG["augment"]:
        x = tf.keras.layers.RandomTranslation(0.0, 0.05)(x)
        x = tf.keras.layers.RandomZoom(0.0, 0.1)(x)
        x = tf.keras.layers.RandomContrast(0.1)(x)

    x = tf.keras.applications.vgg16.preprocess_input(x)
    full_vgg = tf.keras.applications.VGG16(
        include_top=False,
        weights="imagenet",
        input_shape=(height, width, 3)
    )
    full_vgg.trainable = False
    blocks = VGG16_CONFIG["vgg_blocks"]
    feature_extractor = tf.keras.Model(
        full_vgg.input,
        full_vgg.get_layer(f"block{blocks}_pool").output,
        name=f"vgg16_first_{blocks}_blocks"
    )
    x = feature_extractor(x, training=False)
    x = tf.keras.layers.Conv2D(
        128,
        VGG16_CONFIG["kernel"],
        padding="same",
        activation="relu",
        name="additional_conv"
    )(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    if VGG16_CONFIG["dropout"] > 0:
        x = tf.keras.layers.Dropout(VGG16_CONFIG["dropout"])(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)
    return compile_model(model, VGG16_CONFIG["lr"], n_train)

## 8. Run all seeds

For each seed:
1. Create the shared train/validation/test split.
2. Fit the ANN, CNN and VGG16 from scratch.
3. Select neural-network weights using validation accuracy and early stopping.
4. Evaluate each model once on that seed's test set.
5. Save test predictions and metrics.

This means 15 training runs in total (5 seeds × 3 models).

In [ ]:
def calculate_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=np.arange(NUM_CLASSES),
        average="macro",
        zero_division=0
    )
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision),
        "recall_macro": float(recall),
        "f1_macro": float(f1),
    }


# Progress files are stored on Google Drive. The metrics file acts as the
# completion marker: it is written only AFTER that model's predictions.
METRICS_PROGRESS_PATH = OUTPUT_DIR / "multiseed_metrics_progress.csv"
PREDICTIONS_PROGRESS_PATH = OUTPUT_DIR / "multiseed_predictions_progress.csv"
SPLITS_PROGRESS_PATH = OUTPUT_DIR / "multiseed_splits_progress.csv"


def atomic_save_csv(dataframe, destination):
    """Write a temporary CSV first, then replace the previous progress file."""
    temporary_path = destination.with_name(destination.stem + "_temporary" + destination.suffix)
    dataframe.to_csv(temporary_path, index=False)
    os.replace(temporary_path, destination)


# Load previous progress so a restarted Colab session can continue.
if METRICS_PROGRESS_PATH.exists():
    metrics_df_progress = pd.read_csv(METRICS_PROGRESS_PATH)
else:
    metrics_df_progress = pd.DataFrame()

if PREDICTIONS_PROGRESS_PATH.exists():
    predictions_df_progress = pd.read_csv(PREDICTIONS_PROGRESS_PATH)
else:
    predictions_df_progress = pd.DataFrame()

# Recreate and save all split assignments. This is deterministic for each seed.
split_records = []
for seed in SEEDS:
    split = create_seed_split(seed)
    split_records.append(pd.DataFrame({
        "song_id": np.concatenate([
            split["train_ids"], split["val_ids"], split["test_ids"]
        ]),
        "split": (
            ["train"] * len(split["train_ids"])
            + ["validation"] * len(split["val_ids"])
            + ["test"] * len(split["test_ids"])
        ),
        "seed": seed
    }))
atomic_save_csv(pd.concat(split_records, ignore_index=True), SPLITS_PROGRESS_PATH)


def is_model_complete(seed, model_name):
    if metrics_df_progress.empty:
        return False
    return bool(
        ((metrics_df_progress["seed"] == seed)
         & (metrics_df_progress["model"] == model_name)).any()
    )


def save_completed_model(seed, model_name, y_true, y_pred, song_ids, extra_metrics=None):
    """Persist predictions first and metrics second (completion marker)."""
    global metrics_df_progress, predictions_df_progress

    metric_row = {
        "seed": seed,
        "model": model_name,
        **calculate_metrics(y_true, y_pred)
    }
    if extra_metrics:
        metric_row.update(extra_metrics)

    new_predictions = pd.DataFrame({
        "seed": seed,
        "model": model_name,
        "song_id": song_ids,
        "y_true": y_true,
        "y_pred": y_pred
    })

    # Avoid duplicate rows if this cell is re-run after a partial interruption.
    if not predictions_df_progress.empty:
        predictions_df_progress = predictions_df_progress[
            ~(
                (predictions_df_progress["seed"] == seed)
                & (predictions_df_progress["model"] == model_name)
            )
        ]
    predictions_df_progress = pd.concat(
        [predictions_df_progress, new_predictions], ignore_index=True
    )
    atomic_save_csv(predictions_df_progress, PREDICTIONS_PROGRESS_PATH)

    if not metrics_df_progress.empty:
        metrics_df_progress = metrics_df_progress[
            ~(
                (metrics_df_progress["seed"] == seed)
                & (metrics_df_progress["model"] == model_name)
            )
        ]
    metrics_df_progress = pd.concat(
        [metrics_df_progress, pd.DataFrame([metric_row])],
        ignore_index=True
    )
    atomic_save_csv(metrics_df_progress, METRICS_PROGRESS_PATH)

    print(
        f"Saved checkpoint: seed={seed}, model={model_name}. "
        f"Completed models: {len(metrics_df_progress)} / {len(SEEDS) * 3}"
    )


print("Already completed model runs:", len(metrics_df_progress), "/", len(SEEDS) * 3)
if not metrics_df_progress.empty:
    display(metrics_df_progress.sort_values(["seed", "model"]).round(4))


for seed in SEEDS:
    print("\\n" + "=" * 80)
    print(f"SEED {seed}")
    print("=" * 80)

    split = create_seed_split(seed)
    X_train = X_by_id.loc[split["train_ids"]].copy()
    X_val = X_by_id.loc[split["val_ids"]].copy()
    X_test = X_by_id.loc[split["test_ids"]].copy()
    y_train = split["train_labels"]
    y_val = split["val_labels"]
    y_test = split["test_labels"]

    # ---------------- ANN ----------------
    if is_model_complete(seed, "ANN"):
        print(f"[Seed {seed}] ANN already saved — skipping.")
    else:
        print(f"\\n[Seed {seed}] Training ANN")
        ann = build_ann(seed)
        ann.fit(X_train, y_train)
        ann_pred = ann.predict(X_test).astype(int)
        ann_metrics = calculate_metrics(y_test, ann_pred)
        print("ANN test metrics:", ann_metrics)
        print("ANN iterations:", ann.named_steps["model"].n_iter_)
        save_completed_model(seed, "ANN", y_test, ann_pred, split["test_ids"])
        del ann
        gc.collect()

    # ---------------- CNN ----------------
    if is_model_complete(seed, "CNN"):
        print(f"[Seed {seed}] CNN already saved — skipping.")
    else:
        print(f"\\n[Seed {seed}] Training CNN")
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        cnn_train_ds = make_dataset(
            split["train_paths"], y_train, CNN_CONFIG["size"], seed, training=True
        )
        cnn_val_ds = make_dataset(
            split["val_paths"], y_val, CNN_CONFIG["size"], seed, training=False
        )
        cnn_test_ds = make_dataset(
            split["test_paths"], y_test, CNN_CONFIG["size"], seed, training=False
        )
        cnn = build_cnn(seed, len(y_train))
        cnn_history, cnn_best_epoch = fit_with_validation(
            cnn, cnn_train_ds, cnn_val_ds
        )
        cnn_probabilities = cnn.predict(cnn_test_ds, verbose=0)
        cnn_pred = np.argmax(cnn_probabilities, axis=1).astype(int)
        cnn_metrics = calculate_metrics(y_test, cnn_pred)
        print("CNN test metrics:", cnn_metrics, "| Best epoch:", cnn_best_epoch)
        save_completed_model(
            seed, "CNN", y_test, cnn_pred, split["test_ids"],
            extra_metrics={"best_epoch": int(cnn_best_epoch)}
        )
        del cnn, cnn_history, cnn_train_ds, cnn_val_ds, cnn_test_ds
        tf.keras.backend.clear_session()
        gc.collect()

    # ---------------- VGG16 ----------------
    if is_model_complete(seed, "VGG16"):
        print(f"[Seed {seed}] VGG16 already saved — skipping.")
    else:
        print(f"\\n[Seed {seed}] Training VGG16")
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed)
        vgg_train_ds = make_dataset(
            split["train_paths"], y_train, VGG16_CONFIG["size"], seed, training=True
        )
        vgg_val_ds = make_dataset(
            split["val_paths"], y_val, VGG16_CONFIG["size"], seed, training=False
        )
        vgg_test_ds = make_dataset(
            split["test_paths"], y_test, VGG16_CONFIG["size"], seed, training=False
        )
        vgg = build_vgg16(seed, len(y_train))
        vgg_history, vgg_best_epoch = fit_with_validation(
            vgg, vgg_train_ds, vgg_val_ds
        )
        vgg_probabilities = vgg.predict(vgg_test_ds, verbose=0)
        vgg_pred = np.argmax(vgg_probabilities, axis=1).astype(int)
        vgg_metrics = calculate_metrics(y_test, vgg_pred)
        print("VGG16 test metrics:", vgg_metrics, "| Best epoch:", vgg_best_epoch)
        save_completed_model(
            seed, "VGG16", y_test, vgg_pred, split["test_ids"],
            extra_metrics={"best_epoch": int(vgg_best_epoch)}
        )
        del vgg, vgg_history, vgg_train_ds, vgg_val_ds, vgg_test_ds
        tf.keras.backend.clear_session()
        gc.collect()

print("\\nMulti-seed training loop finished.")
print("Completed model runs:", len(metrics_df_progress), "/", len(SEEDS) * 3)
display(metrics_df_progress.sort_values(["seed", "model"]).round(4))

## 9. Results for each seed

The table below reports the four test metrics for each model and seed. Accuracy is the proportion of correctly classified songs. Macro precision, recall and F1 give equal weight to each genre.

In [ ]:
# Load the durable progress files rather than relying on variables kept in RAM.
# This also works if the training cell was interrupted and the runtime was restarted.
metrics_df = pd.read_csv(METRICS_PROGRESS_PATH)
predictions_df = pd.read_csv(PREDICTIONS_PROGRESS_PATH)

display(metrics_df.sort_values(["seed", "model"]).round(4))

# Publish final-named files only when all 15 model/seed combinations are present.
expected_runs = len(SEEDS) * 3
completed_runs = set(zip(metrics_df["seed"].astype(int), metrics_df["model"]))
expected_pairs = {
    (int(seed), model_name)
    for seed in SEEDS
    for model_name in ["ANN", "CNN", "VGG16"]
}
missing_runs = sorted(expected_pairs - completed_runs)

if missing_runs:
    print("The experiment is not complete yet. Missing runs:", missing_runs)
    print("Progress is safe in the *_progress.csv files. Re-run the training cell to continue.")
else:
    metrics_df.to_csv(OUTPUT_DIR / "multiseed_metrics.csv", index=False)
    predictions_df.to_csv(OUTPUT_DIR / "multiseed_predictions.csv", index=False)
    pd.read_csv(SPLITS_PROGRESS_PATH).to_csv(
        OUTPUT_DIR / "multiseed_splits.csv", index=False
    )
    print("All 15 runs are complete. Final result files saved to:", OUTPUT_DIR)

## 10. Mean and standard deviation across seeds

The mean summarizes average performance across the five splits. The standard deviation describes how much the result changes from one split to another. A lower standard deviation indicates more stable results in this experiment.

These descriptive statistics should not be interpreted as a formal significance test.

In [ ]:
metric_columns = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

summary_df = (
    metrics_df
    .groupby("model")[metric_columns]
    .agg(["mean", "std"])
)

display(summary_df.round(4))
summary_df.to_csv(OUTPUT_DIR / "multiseed_mean_std.csv")

# A compact table is easier to use in the report.
compact_summary = metrics_df.groupby("model")[metric_columns].agg(["mean", "std"])
for metric in metric_columns:
    compact_summary[(metric, "mean_std")] = (
        compact_summary[(metric, "mean")].map(lambda x: f"{x:.3f}")
        + " ± "
        + compact_summary[(metric, "std")].map(lambda x: f"{x:.3f}")
    )
display(compact_summary[[(m, "mean_std") for m in metric_columns]])

## 11. Visual comparison across seeds

Each point represents one seed. The horizontal line and error bar show the mean and one standard deviation. The plot helps distinguish a model with a higher average from one whose performance changes more strongly across splits.

In [ ]:
models_order = ["ANN", "CNN", "VGG16"]
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, metric, title in [
    (axes[0], "accuracy", "Test accuracy across seeds"),
    (axes[1], "f1_macro", "Macro F1 across seeds")
]:
    for position, model_name in enumerate(models_order):
        values = metrics_df.loc[metrics_df["model"] == model_name, metric].to_numpy()
        ax.scatter(
            np.full(len(values), position),
            values,
            label=model_name if ax is axes[0] else None
        )
        mean_value = np.mean(values)
        std_value = np.std(values, ddof=1)
        ax.errorbar(
            position, mean_value, yerr=std_value,
            fmt="s", capsize=5
        )
    ax.set_xticks(range(len(models_order)))
    ax.set_xticklabels(models_order)
    ax.set_ylim(0, 1)
    ax.set_ylabel(metric)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

# Save the figure for later use in the report.
fig.savefig(OUTPUT_DIR / "multiseed_accuracy_f1.png", dpi=200, bbox_inches="tight")

## 12. Inspect the confusion matrices for each seed

The following cell creates a confusion matrix for every model and seed. Rows are the true genres and columns are the predicted genres. Values are normalized by true genre, making the rows easier to compare.

These matrices are useful for checking whether the same genre confusions appear repeatedly, rather than only in one split.

In [ ]:
predictions_df = pd.read_csv(PREDICTIONS_PROGRESS_PATH)

if predictions_df.empty:
    raise RuntimeError(
        "No completed model predictions found yet. Run the training cell first."
    )

for seed in SEEDS:
    seed_predictions = predictions_df[predictions_df["seed"] == seed]
    for model_name in models_order:
        subset = seed_predictions[seed_predictions["model"] == model_name]
        matrix = confusion_matrix(
            subset["y_true"],
            subset["y_pred"],
            labels=np.arange(NUM_CLASSES),
            normalize="true"
        )

        fig, ax = plt.subplots(figsize=(8, 7))
        ConfusionMatrixDisplay(
            matrix,
            display_labels=class_names
        ).plot(
            ax=ax,
            xticks_rotation=45,
            colorbar=True,
            values_format=".2f"
        )
        ax.set_title(f"{model_name} — normalized confusion matrix — seed {seed}")
        ax.set_xlabel("Predicted genre")
        ax.set_ylabel("True genre")
        plt.tight_layout()
        fig.savefig(
            OUTPUT_DIR / f"confusion_{model_name.lower()}_seed_{seed}.png",
            dpi=160,
            bbox_inches="tight"
        )
        plt.show()
        plt.close(fig)

## 13. Interpretation notes

Use the results to answer these questions:

- **Consistency:** Does the same model have the highest mean accuracy and macro F1 across the five seeds?
- **Sensitivity to the split:** Which model has the largest standard deviation?
- **Stability of genre errors:** Do the same off-diagonal confusion patterns appear across seeds?
- **Limitations:** Five seeds are more informative than one, but still a small sample. Differences in means are descriptive; do not claim statistical significance based only on this experiment.

The three winning configurations were selected using the original seed-42 experiments. Keeping them fixed here makes the multi-seed experiment practical, but it does not repeat hyperparameter selection independently for every seed. Therefore, the results measure the stability of these selected configurations, not the stability of the full model-selection process.

All CSV files, predictions, split assignments and plots are stored in:

`/content/drive/MyDrive/MLSM/Results/Lab 2/multiseed_evaluation_v1`